# 01. Prompt Engineering 🏪

## 학습 목표
- 좋은 프롬프트의 조건을 이해하고 체계적으로 작성
- Zero-shot, Few-shot, Chain-of-Thought 패턴 실습
- 출력 형식(JSON, 리스트) 제어 기법 습득
- System Prompt 설계 원칙 학습
- ai-ipsonum 매장 추출 프롬프트 A/B 테스트 프레임워크 구축

## 참고 자료
- [Chain-of-Thought Prompting Elicits Reasoning](https://arxiv.org/abs/2201.11903) (Wei et al., 2022)
- [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering)

## ai-ipsonum 연계 🏪
- 매장 추출 정확도를 높이는 프롬프트 A/B 테스트
- 참고: `ai-ipsonum/src/lib/ai-engines/query-templates.ts`

---

In [ ]:
import json
import re
import textwrap
from collections import Counter
from typing import Any

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import numpy as np
import pandas as pd

## 1. Prompt Engineering 기초: 좋은 프롬프트의 조건

LLM은 입력된 텍스트(프롬프트)에 따라 출력이 크게 달라진다.  
동일한 작업이라도 프롬프트 설계에 따라 정확도, 출력 형식, 일관성이 크게 달라진다.

### 좋은 프롬프트의 4가지 원칙

| 원칙 | 설명 | 예시 |
|------|------|------|
| **명확성 (Clarity)** | 모호하지 않게, 구체적으로 | "요약해줘" → "3문장 이내로 요약해줘" |
| **구체성 (Specificity)** | 원하는 출력 형식과 범위 지정 | "리스트로 정리해줘" → "JSON 배열로 리턴해줘" |
| **맥락 (Context)** | 필요한 배경 정보 제공 | "이 텍스트는 네이버 지도 리뷰입니다" |
| **제약조건 (Constraints)** | 불필요한 출력 억제 | "추측이나 가정은 하지 마" |

In [ ]:
# ---- Mock LLM: 프롬프트의 특성에 따라 다른 응답을 시뮬레이션 ----

def mock_llm(prompt: str, system: str = "") -> str:
    """
    API 없이 동작하는 Mock LLM.
    프롬프트의 키워드에 따라 미리 정의된 응답을 반환한다.
    """
    prompt_lower = prompt.lower()
    
    # 매장 추출 관련
    if "매장" in prompt_lower or "카페" in prompt_lower or "맛집" in prompt_lower:
        if "json" in prompt_lower:
            return json.dumps([
                {"name": "블루보틀 성수점", "rank": 1, "category": "카페"},
                {"name": "스타벅스 종로점", "rank": 2, "category": "카페"},
                {"name": "컨테이너 성수동점", "rank": 3, "category": "카페"}
            ], ensure_ascii=False, indent=2)
        else:
            return (
                "성수동 인기 카페 TOP 3\n"
                "1. 블루보틀 성수점 - 스페셜티 커피로 유명\n"
                "2. 스타벅스 종로점 - 넓은 좌석, 접근성 좋음\n"
                "3. 컨테이너 성수동점 - 분위기 좋은 로스터리"
            )
    
    # 감정 분석 관련
    if "감정" in prompt_lower or "sentiment" in prompt_lower:
        if "json" in prompt_lower:
            return json.dumps({"sentiment": "positive", "confidence": 0.92}, ensure_ascii=False)
        return "긍정적입니다."
    
    # 단계별 추론 관련
    if "step by step" in prompt_lower or "단계" in prompt_lower:
        return (
            "Step 1: 문제를 파악합니다.\n"
            "Step 2: 관련 정보를 수집합니다.\n"
            "Step 3: 분석 결과를 정리합니다.\n"
            "따라서 답은 42입니다."
        )
    
    return "요청을 처리했습니다. 더 구체적인 지시를 주시면 더 좋은 결과를 드릴 수 있습니다."


# 테스트
print("=== 모호한 프롬프트 ===")
print(mock_llm("카페 알려줘"))
print()
print("=== 구체적인 프롬프트 ===")
print(mock_llm("성수동 인기 카페 TOP 3를 JSON 배열로 알려줘"))

---
## 2. Zero-shot vs Few-shot: 예시 개수에 따른 성능 변화

### Zero-shot
예시 없이 작업만 설명. LLM의 사전 학습된 지식에 의존한다.

### Few-shot
입력-출력 예시를 몇 개 제공하여 패턴을 학습시킨다.

| 방법 | 예시 수 | 장점 | 단점 |
|------|---------|------|------|
| Zero-shot | 0개 | 간단, 토큰 절약 | 복잡한 패턴 인식 어려움 |
| One-shot | 1개 | 최소한의 패턴 전달 | 예시가 편향될 수 있음 |
| Few-shot | 2~5개 | 안정적인 패턴 인식 | 토큰 사용량 증가 |

In [ ]:
# Zero-shot vs Few-shot 비교 예시: 감정 분석

# --- Zero-shot ---
zero_shot_prompt = """
다음 리뷰의 감정을 분석해줘.

리뷰: "커피가 정말 맛있어요! 다만 자리가 좀 좋았으면..."
"""

# --- Few-shot ---
few_shot_prompt = """
다음 리뷰의 감정을 분석해줘. 형식: {"sentiment": "positive/negative/mixed", "confidence": 0.0~1.0}

예시 1:
리뷰: "음식도 맛있고 서비스도 친절해요!"
결과: {"sentiment": "positive", "confidence": 0.95}

예시 2:
리뷰: "맛은 보통인데 가격이 너무 비싸네요"
결과: {"sentiment": "negative", "confidence": 0.7}

예시 3:
리뷰: "커피는 최고인데 디저트는 별로에요"
결과: {"sentiment": "mixed", "confidence": 0.8}

리뷰: "커피가 정말 맛있어요! 다만 자리가 좀 좋았으면..."
결과:
"""

print("=== Zero-shot Prompt ===")
print(zero_shot_prompt.strip())
print("\n\u2192 예상 응답: \"\uae0d\uc815\uc801\uc785\ub2c8\ub2e4\" (형식 불일정)")

print("\n" + "="*50)
print("\n=== Few-shot Prompt ===")
print(few_shot_prompt.strip())
print('\n\u2192 \uc608\uc0c1 \uc751\ub2f5: {"sentiment": "mixed", "confidence": 0.75} (\ud615\uc2dd \uc77c\uad00\uc801)')

In [ ]:
# Few-shot 예시 수에 따른 성능 시뮬레이션

def simulate_few_shot_performance(n_examples: int) -> dict:
    """예시 수에 따른 성능을 시뮬레이션 (실제 논문 경향을 반영)"""
    np.random.seed(42)
    # 예시가 많을수록 정확도가 오르지만 수익 체감
    base_accuracy = 0.65
    improvement = 0.30 * (1 - np.exp(-0.8 * n_examples))
    noise = np.random.normal(0, 0.02)
    accuracy = min(base_accuracy + improvement + noise, 0.98)
    
    # 형식 일관성
    format_consistency = min(0.3 + 0.15 * n_examples + np.random.normal(0, 0.03), 1.0)
    
    return {
        "n_examples": n_examples,
        "accuracy": round(accuracy, 3),
        "format_consistency": round(format_consistency, 3),
        "token_cost": 50 + 80 * n_examples  # 예시당 ~80 토큰
    }

results = [simulate_few_shot_performance(n) for n in range(6)]
df = pd.DataFrame(results)
print(df.to_string(index=False))

# 시각화
fig, ax1 = plt.subplots(figsize=(8, 5))

ax1.plot(df["n_examples"], df["accuracy"], 'b-o', label="Accuracy", linewidth=2)
ax1.plot(df["n_examples"], df["format_consistency"], 'g-s', label="Format Consistency", linewidth=2)
ax1.set_xlabel("Number of Few-shot Examples")
ax1.set_ylabel("Score")
ax1.set_ylim(0, 1.1)
ax1.legend(loc='center left')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.bar(df["n_examples"], df["token_cost"], alpha=0.2, color='red', label="Token Cost")
ax2.set_ylabel("Token Cost")
ax2.legend(loc='center right')

plt.title("Few-shot: Examples vs Performance vs Cost")
plt.tight_layout()
plt.show()

---
## 3. Chain-of-Thought (CoT): 단계별 추론 유도

LLM에게 "단계별로 생각해봐" 라고 지시하면 복잡한 추론 능력이 크게 향상된다.

### CoT 유형

1. **Zero-shot CoT**: `"단계별로 생각해보세요"` 한 문장 추가
2. **Few-shot CoT**: 추론 과정이 포함된 예시 제공
3. **Self-Consistency**: 여러 번 CoT 수행 후 다수결 투표

### 예시: 일반 vs CoT

**일반 프롬프트**: "15% 할인된 20,000원짜리 커피 3잔의 총 가격은?"  
→ LLM이 단계를 건너뛰고 잘못된 답을 낼 수 있음

**CoT 프롬프트**: "단계별로 계산해줘. 15% 할인된 20,000원짜리 커피 3잔의 총 가격은?"  
→ Step 1: 할인액 = 20,000 × 0.15 = 3,000  
→ Step 2: 할인가 = 20,000 - 3,000 = 17,000  
→ Step 3: 총액 = 17,000 × 3 = **51,000원**

In [ ]:
# CoT 패턴 구현 예시

def solve_with_cot(question: str) -> str:
    """
    Chain-of-Thought 추론을 시뮬레이션하는 함수.
    실제로는 LLM이 이 과정을 수행하지만, 여기서는 구조를 보여주기 위해 직접 구현.
    """
    steps = []
    
    # 예시: 할인 계산
    if "할인" in question:
        original_price = 20000
        discount_rate = 0.15
        quantity = 3
        
        steps.append(f"Step 1: 할인액 계산 = {original_price:,} x {discount_rate} = {original_price * discount_rate:,.0f}원")
        discount_amount = original_price * discount_rate
        
        steps.append(f"Step 2: 할인가 계산 = {original_price:,} - {discount_amount:,.0f} = {original_price - discount_amount:,.0f}원")
        discounted_price = original_price - discount_amount
        
        steps.append(f"Step 3: 총액 계산 = {discounted_price:,.0f} x {quantity} = {discounted_price * quantity:,.0f}원")
        total = discounted_price * quantity
        
        steps.append(f"\n따라서 답은 {total:,.0f}원입니다.")
    
    return "\n".join(steps)


question = "15% 할인된 20,000원짜리 커피 3잔의 총 가격은?"

print("=== 일반 프롬프트 ===")
print(f"Q: {question}")
print("A: 51,000원 (단순 답만)")

print("\n=== CoT 프롬프트 ===")
print(f"Q: 단계별로 계산해줘. {question}")
print(solve_with_cot(question))

In [ ]:
# Self-Consistency: 여러 번 CoT를 수행하고 다수결 투표

def self_consistency_simulation(n_samples: int = 5) -> dict:
    """
    Self-Consistency 시뮬레이션.
    LLM을 n번 호출하여 가장 많이 나온 답을 선택.
    """
    np.random.seed(42)
    # 실제로는 LLM이 temperature > 0으로 여러 번 응답
    possible_answers = [51000, 51000, 51000, 45000, 51000]  # 대부분 맞지만 가끔 틀림
    samples = possible_answers[:n_samples]
    
    counter = Counter(samples)
    majority = counter.most_common(1)[0]
    
    return {
        "samples": samples,
        "vote_counts": dict(counter),
        "final_answer": majority[0],
        "confidence": majority[1] / n_samples
    }

result = self_consistency_simulation(5)
print("Self-Consistency 결과:")
print(f"  각 샘플 답: {result['samples']}")
print(f"  투표 결과: {result['vote_counts']}")
print(f"  최종 답: {result['final_answer']:,}원 (confidence: {result['confidence']:.0%})")

---
## 4. Output Format 제어: JSON, 리스트, 구조화된 출력 유도

LLM 출력을 프로그래밍적으로 활용하려면 **구조화된 출력**이 필수.

### 전략

1. **예시 제공**: 원하는 형식의 예시를 보여주는 것이 가장 효과적
2. **스키마 명시**: JSON 키 이름과 타입을 맨 위에 정의
3. **역할 부여**: "JSON만 출력해줘. 설명 없이."
4. **검증 로직**: 파싱 실패 시 재시도

In [ ]:
# Output Format 제어 예시

# --- 프롬프트 템플릿: JSON 출력 유도 ---
json_prompt_template = """
다음 리뷰를 분석하여 아래 JSON 형식으로만 응답해줘. 설명은 포함하지 마.

형식:
{{
  "store_name": "매장명",
  "sentiment": "positive | negative | mixed",
  "rating": 1~5,
  "keywords": ["키워드1", "키워드2"]
}}

리뷰: "{review}"
"""

# 테스트 리뷰들
test_reviews = [
    "블루보틀 성수점 커피가 정말 훌륭해요! 분위기도 좋고 스페셜티가 맛있어요.",
    "스타벅스는 어디나 비슷해요. 가격 대비 별로인 듯.",
    "컨테이너 커피 맛은 좋은데 웨이팅이 너무 길어요. 30분 기다렸어요."
]

# Mock 응답 시뮬레이션
mock_responses = [
    {"store_name": "블루보틀 성수점", "sentiment": "positive", "rating": 5, "keywords": ["커피", "분위기", "스페셜티"]},
    {"store_name": "스타벅스", "sentiment": "negative", "rating": 2, "keywords": ["가격", "별로"]},
    {"store_name": "컨테이너", "sentiment": "mixed", "rating": 3, "keywords": ["맛", "웨이팅", "대기시간"]}
]

for review, response in zip(test_reviews, mock_responses):
    prompt = json_prompt_template.format(review=review)
    print(f"리뷰: \"{review[:30]}...\"")
    print(f"응답: {json.dumps(response, ensure_ascii=False)}")
    
    # JSON 파싱 검증
    try:
        parsed = json.loads(json.dumps(response))
        assert "store_name" in parsed
        assert parsed["sentiment"] in ["positive", "negative", "mixed"]
        print("✓ 파싱 성공")
    except (json.JSONDecodeError, AssertionError) as e:
        print(f"✗ 파싱 실패: {e}")
    print()

In [ ]:
# 리스트 출력 유도 예시

list_prompt = """
성수동 카페 TOP 5를 아래 형식으로 알려줘:

1. [\ub9e4\uc7a5\uba85] - [\ud55c\uc904 \uc124\uba85] (\uce74\ud14c\uace0\ub9ac: \uce74\ud398/\ub85c\uc2a4\ud130\ub9ac/...)
2. ...
"""

mock_list_response = """1. 블루보틀 성수점 - 스페셜티 커피와 미니말리즘 인테리어 (카테고리: 카페)
2. 스타벅스 종로점 - 넓은 좌석과 다양한 메뉴 (카테고리: 카페)
3. 컨테이너 성수동 - 루프탑 있는 로스터리 (카테고리: 로스터리)
4. 솔길체 - 단체 모임에 좋은 넓은 공간 (카테고리: 카페)
5. 플릿화이트 - 바다 바리스타 스타일 (카테고리: 카페)"""

print("=== 리스트 출력 프롬프트 ===")
print(list_prompt.strip())
print("\n=== Mock 응답 ===")
print(mock_list_response)

# 파싱
pattern = r"(\d+)\. (.+?) - (.+?) \(카테고리: (.+?)\)"
matches = re.findall(pattern, mock_list_response)
print("\n=== 파싱 결과 ===")
for rank, name, desc, category in matches:
    print(f"  #{rank} {name} [{category}]: {desc}")

---
## 5. System Prompt 설계: 역할 부여, 제약 조건 설정

System Prompt는 LLM의 **전체 행동 방침**을 설정한다.  
User Prompt보다 우선순위가 높고, 모든 대화에 적용된다.

### System Prompt 설계 원칙

| 요소 | 설명 | 예시 |
|------|------|------|
| **역할 (Role)** | LLM에게 페르소나 부여 | "당신은 매장 추출 전문가입니다" |
| **목적 (Goal)** | 해야 할 일 명시 | "리뷰에서 매장 정보를 추출하세요" |
| **형식 (Format)** | 출력 형식 지정 | "JSON 배열로 출력하세요" |
| **제약 (Constraints)** | 하지 말아야 할 것 | "추측하지 마세요" |
| **예시 (Examples)** | 기대 동작 예시 | "예시: ... → ..." |

In [ ]:
# System Prompt 설계 예시

# --- 나쁜 예시: 너무 모호한 System Prompt ---
bad_system_prompt = """
당신은 도움이 되는 어시스턴트입니다.
"""

# --- 좋은 예시: 구체적인 System Prompt ---
good_system_prompt = """
당신은 한국 로컸 매장 정보 추출 전문가입니다.

## 역할
사용자가 지역과 카테고리를 제공하면, 해당 지역의 인기 매장을 추출합니다.

## 출력 형식
반드시 JSON 배열로 응답하세요. 각 객체는 다음 필드를 포함합니다:
- name: 매장명 (string)
- rank: 순위 (number, 1부터)
- category: 카테고리 (string)
- reason: 추천 이유 (string, 1문장)

## 제약 조건
- 확실하지 않은 매장은 포함하지 마세요
- JSON 외 텍스트는 출력하지 마세요
- 최대 10개까지만 출력하세요
"""

print("=== 나쁜 System Prompt ===")
print(bad_system_prompt.strip())
print("\u2192 문제: 역할, 형식, 제약조건 없음. LLM이 자유롭게 응답.")

print("\n" + "="*50)
print("\n=== 좋은 System Prompt ===")
print(good_system_prompt.strip())
print("\n\u2192 장점: 역할/형식/제약조건 명확. 일관된 출력 기대 가능.")

In [ ]:
# System Prompt 테스트: 동일한 쿼리에 다른 System Prompt 적용

user_query = "성수동 인기 카페 TOP 3"

# 나쁜 System Prompt 적용 시 예상 응답 (Mock)
bad_response = """성수동에는 다양한 카페가 있습니다. 
블루보틀이 유명하고, 스타벅스도 있습니다. 
컨테이너 커피도 추천합니다!"""

# 좋은 System Prompt 적용 시 예상 응답 (Mock)
good_response = json.dumps([
    {"name": "블루보틀 성수점", "rank": 1, "category": "카페", "reason": "스페셜티 드립 커피로 유명"},
    {"name": "스타벅스 종로점", "rank": 2, "category": "카페", "reason": "접근성과 넓은 좌석"},
    {"name": "컨테이너 성수동점", "rank": 3, "category": "로스터리", "reason": "루프탑 있는 분위기"}
], ensure_ascii=False, indent=2)

print("Query:", user_query)
print("\n--- Bad System Prompt 응답 ---")
print(bad_response)
print("\u2192 문제: 비구조화, 파싱 어려움, 순위 없음")

print("\n--- Good System Prompt 응답 ---")
print(good_response)
print("\u2192 장점: JSON 파싱 가능, 필드 일관성, 프로그래밍 활용 가능")

---
## 6. 🏪 매장 추출 프롬프트 A/B 테스트

ai-ipsonum에서 실제로 사용하는 매장 추출 프롬프트를 **A/B 테스트**하는 프레임워크.

### 평가 지표
1. **추출 정확도 (Extraction Accuracy)**: 정답 매장을 얼마나 정확하게 추출했는가
2. **형식 일관성 (Format Consistency)**: JSON 파싱 성공률, 필드 완전성
3. **토큰 효율성**: 동일 결과를 얻기 위한 토큰 사용량

In [ ]:
# --- A/B 테스트용 프롬프트 정의 ---

prompt_a = """
{area} 지역의 인기 {category}를 알려줘.
"""

prompt_b = """
당신은 한국 로컸 매장 정보 전문가입니다.

{area} 지역의 인기 {category} TOP 5를 아래 JSON 형식으로 알려주세요.

출력 형식:
[
  {{"name": "매장명", "rank": 1, "category": "카테고리", "reason": "추천 이유"}}
]

제약조건:
- 확실하지 않은 매장은 제외
- JSON만 출력 (설명 없이)
- 각 매장의 reason은 1문장 이내
"""

print("=== Prompt A (현재 / 단순) ===")
print(prompt_a.format(area="성수동", category="카페").strip())
print("\n=== Prompt B (개선 / 구조화) ===")
print(prompt_b.format(area="성수동", category="카페").strip())

In [ ]:
# --- 샘플 데이터로 A/B 테스트 시뮬레이션 ---

# 정답 데이터 (Ground Truth)
ground_truth = [
    {"name": "블루보틀 성수점", "category": "카페"},
    {"name": "스타벅스 종로점", "category": "카페"},
    {"name": "컨테이너 성수동", "category": "카페"},
    {"name": "솔길체", "category": "카페"},
    {"name": "플릿화이트", "category": "카페"},
]

# Prompt A 응답 시뮬레이션 (5회 반복)
prompt_a_responses = [
    # 답변 1: 비구조화 텍스트
    {
        "raw": "성수동 카페 추천! 1. 블루보틀 2. 스타벅스 3. 매머드 카페",
        "parsed": ["블루보틀 성수점", "스타벅스 종로점", "매머드 카페"],
        "json_valid": False
    },
    # 답변 2: 일부 JSON
    {
        "raw": '[{"name": "블루보틀"}, {"name": "컨테이너"}]',
        "parsed": ["블루보틀 성수점", "컨테이너 성수동"],
        "json_valid": True
    },
    # 답변 3: 비구조화
    {
        "raw": "성수동 카페\n- 블루보틀\n- 스타벅스\n- 컨테이너\n- 솔길체",
        "parsed": ["블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "솔길체"],
        "json_valid": False
    },
    # 답변 4: hallucination 포함
    {
        "raw": "1. 블루보틀 2. 스타벅스 3. 매머드 4. 리보 카페",
        "parsed": ["블루보틀 성수점", "스타벅스 종로점", "매머드 카페", "리보 카페"],
        "json_valid": False
    },
    # 답변 5
    {
        "raw": "블루보틀, 스타벅스, 컨테이너, 플릿화이트",
        "parsed": ["블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "플릿화이트"],
        "json_valid": False
    }
]

# Prompt B 응답 시뮬레이션 (5회 반복)
prompt_b_responses = [
    {
        "raw": json.dumps([{"name": "블루보틀 성수점", "rank": 1, "category": "카페", "reason": "스페셜티 커피"}, {"name": "스타벅스 종로점", "rank": 2, "category": "카페", "reason": "접근성"}, {"name": "컨테이너 성수동", "rank": 3, "category": "카페", "reason": "루프탑"}, {"name": "솔길체", "rank": 4, "category": "카페", "reason": "단체석"}, {"name": "플릿화이트", "rank": 5, "category": "카페", "reason": "바리스타"}], ensure_ascii=False),
        "parsed": ["블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "솔길체", "플릿화이트"],
        "json_valid": True
    },
    {
        "raw": json.dumps([{"name": "블루보틀 성수점", "rank": 1, "category": "카페", "reason": "하우스 블렌드"}, {"name": "컨테이너 성수동", "rank": 2, "category": "카페", "reason": "로스터리"}, {"name": "스타벅스 종로점", "rank": 3, "category": "카페", "reason": "다양한 메뉴"}, {"name": "플릿화이트", "rank": 4, "category": "카페", "reason": "트렌디"}, {"name": "솔길체", "rank": 5, "category": "카페", "reason": "넓은 공간"}], ensure_ascii=False),
        "parsed": ["블루보틀 성수점", "컨테이너 성수동", "스타벅스 종로점", "플릿화이트", "솔길체"],
        "json_valid": True
    },
    {
        "raw": json.dumps([{"name": "블루보틀 성수점", "rank": 1, "category": "카페", "reason": "드립 커피"}, {"name": "스타벅스 종로점", "rank": 2, "category": "카페", "reason": "편리함"}, {"name": "컨테이너 성수동", "rank": 3, "category": "카페", "reason": "분위기"}, {"name": "솔길체", "rank": 4, "category": "카페", "reason": "그룹 모임"}], ensure_ascii=False),
        "parsed": ["블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "솔길체"],
        "json_valid": True
    },
    {
        "raw": json.dumps([{"name": "블루보틀 성수점", "rank": 1, "category": "카페", "reason": "스페셜티"}, {"name": "스타벅스 종로점", "rank": 2, "category": "카페", "reason": "접근성"}, {"name": "컨테이너 성수동", "rank": 3, "category": "카페", "reason": "로스터리"}, {"name": "솔길체", "rank": 4, "category": "카페", "reason": "단체석"}, {"name": "플릿화이트", "rank": 5, "category": "카페", "reason": "바리스타"}], ensure_ascii=False),
        "parsed": ["블루보틀 성수점", "스타벅스 종로점", "컨테이너 성수동", "솔길체", "플릿화이트"],
        "json_valid": True
    },
    {
        "raw": json.dumps([{"name": "블루보틀 성수점", "rank": 1, "category": "카페", "reason": "커피 맛"}, {"name": "컨테이너 성수동", "rank": 2, "category": "카페", "reason": "로스터리"}, {"name": "스타벅스 종로점", "rank": 3, "category": "카페", "reason": "다양한 메뉴"}, {"name": "솔길체", "rank": 4, "category": "카페", "reason": "단체 모임"}, {"name": "플릿화이트", "rank": 5, "category": "카페", "reason": "분위기"}], ensure_ascii=False),
        "parsed": ["블루보틀 성수점", "컨테이너 성수동", "스타벅스 종로점", "솔길체", "플릿화이트"],
        "json_valid": True
    }
]

print(f"Prompt A 응답 수: {len(prompt_a_responses)}")
print(f"Prompt B 응답 수: {len(prompt_b_responses)}")

In [ ]:
# --- 평가 함수 ---

def evaluate_extraction(responses: list, ground_truth: list) -> dict:
    """프롬프트 응답을 평가하는 함수"""
    gt_names = {item["name"] for item in ground_truth}
    
    accuracies = []
    json_valid_count = 0
    hallucination_count = 0
    
    for resp in responses:
        parsed_names = set(resp["parsed"])
        
        # Precision: 추출된 것 중 정답인 비율
        true_positives = len(parsed_names & gt_names)
        precision = true_positives / len(parsed_names) if parsed_names else 0
        
        # Recall: 정답 중 추출된 비율
        recall = true_positives / len(gt_names) if gt_names else 0
        
        # F1
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        accuracies.append(f1)
        
        # JSON 유효성
        if resp["json_valid"]:
            json_valid_count += 1
        
        # Hallucination (정답에 없는 매장)
        hallucinations = parsed_names - gt_names
        hallucination_count += len(hallucinations)
    
    return {
        "avg_f1": np.mean(accuracies),
        "json_valid_rate": json_valid_count / len(responses),
        "total_hallucinations": hallucination_count,
        "avg_hallucination_per_response": hallucination_count / len(responses)
    }


eval_a = evaluate_extraction(prompt_a_responses, ground_truth)
eval_b = evaluate_extraction(prompt_b_responses, ground_truth)

print("=== A/B 테스트 결과 ===")
print(f"\n{'Metric':<30} {'Prompt A':>12} {'Prompt B':>12} {'\u0394':>10}")
print("-" * 65)
for metric in ["avg_f1", "json_valid_rate", "avg_hallucination_per_response"]:
    a_val = eval_a[metric]
    b_val = eval_b[metric]
    delta = b_val - a_val
    direction = "+" if delta > 0 else ""
    # hallucination은 낮을수록 좋음
    better = "✓" if (delta > 0 and "hallucination" not in metric) or (delta < 0 and "hallucination" in metric) else ""
    print(f"{metric:<30} {a_val:>12.3f} {b_val:>12.3f} {direction}{delta:>9.3f} {better}")

In [ ]:
# A/B 테스트 결과 시각화

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

metrics = [
    ("Avg F1 Score", eval_a["avg_f1"], eval_b["avg_f1"]),
    ("JSON Valid Rate", eval_a["json_valid_rate"], eval_b["json_valid_rate"]),
    ("Avg Hallucinations", eval_a["avg_hallucination_per_response"], eval_b["avg_hallucination_per_response"]),
]

for ax, (title, val_a, val_b) in zip(axes, metrics):
    bars = ax.bar(["Prompt A\n(simple)", "Prompt B\n(structured)"], [val_a, val_b],
                  color=["#ff6b6b", "#4ecdc4"], edgecolor="black", linewidth=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylim(0, max(val_a, val_b) * 1.3)
    
    for bar, val in zip(bars, [val_a, val_b]):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
                f'{val:.2f}', ha='center', va='bottom', fontweight='bold')
    
    ax.grid(axis='y', alpha=0.3)

plt.suptitle("Prompt A/B Test Results", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n\u2192 Prompt B(구조화된 프롬프트)가 F1, JSON 유효성, Hallucination 모두에서 우수")

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: CoT 프롬프트 작성

다음 문제를 풀기 위한 **Few-shot CoT 프롬프트**를 작성하세요.  
문제: "매장의 네이버 평점(5점 만점)과 리뷰 수를 바탕으로 종합 점수를 계산하라"  
종합 점수 = 평점 × 0.7 + min(리뷰수/100, 1.0) × 0.3

In [ ]:
# TODO: Few-shot CoT 프롬프트를 작성하세요
# 요구사항:
# 1. 추론 과정이 포함된 예시 2개를 제공
# 2. 새로운 매장 데이터(\ud3c9\uc810: 4.2, \ub9ac\ubdf0\uc218: 85)에 대해 추론하도록 유도

cot_prompt = """
# TODO: 여기에 Few-shot CoT 프롬프트를 작성하세요
"""

print(cot_prompt)

### 연습 2: System Prompt 설계

ai-ipsonum의 "맛집 추출" 기능을 위한 System Prompt를 설계하세요.  
요구사항:
- 역할, 목적, 출력 형식, 제약조건을 모두 포함
- 출력 형식은 JSON으로 `{"name", "rank", "category", "price_range", "specialty"}` 포함
- 가격대가 불확실하면 "미정"으로 표시하도록 지시

In [ ]:
# TODO: System Prompt를 설계하세요

restaurant_system_prompt = """
# TODO: 여기에 System Prompt를 작성하세요
"""

print(restaurant_system_prompt)

---
## 핵심 정리

| 개념 | 설명 | 활용 장면 |
|------|------|----------|
| Zero-shot | 예시 없이 지시만 | 단순한 작업, 토큰 절약 |
| Few-shot | 2~5개 예시 제공 | 구조화된 출력, 패턴 인식 |
| Chain-of-Thought | 단계별 추론 유도 | 복잡한 계산/추론 문제 |
| Self-Consistency | 다수결 투표 | 신뢰도 향상, 비용 부담 |
| Output Format 제어 | JSON/리스트 유도 | API 연동, 데이터 파이프라인 |
| System Prompt | 역할/제약/형식 설정 | 일관된 동작, 프로덕션 시스템 |
| A/B 테스트 | 프롬프트 비교 평가 | 프롬프트 최적화 |

**다음 노트북**: [02-structured-extraction.ipynb](02-structured-extraction.ipynb) - 구조화된 데이터 추출